In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [ ]:
load_dotenv(override=True )

In [ ]:
import os

ollamaApiKey = os.getenv("OLLAMA_API_KEY")
ollamaBaseUrl = os.getenv("OLLAMA_BASE_URL")
print(ollamaBaseUrl, ollamaApiKey)

In [ ]:
openApiClient = OpenAI(base_url=ollamaBaseUrl, api_key=ollamaApiKey)

In [ ]:
reader = PdfReader("Jimmy/professional_work_experience.pdf")
professionalBackgroundContent = ""
for page in reader.pages:
    extractedText = page.extract_text()
    professionalBackgroundContent += extractedText
print(professionalBackgroundContent)

In [ ]:
summary = ""
with open("Jimmy/summary.txt") as f:
    summary = f.read()
print(summary)

In [ ]:
name = "Tarunpreet Singh"

In [ ]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be informal and engaging, as if talking to a friend who found you while browsing the internet. \
If you don't know the answer, be friendly and change the topic after politely telling that you do not know the answer."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{professionalBackgroundContent}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [ ]:
system_prompt

In [ ]:
def parseResponseByRemovingTag(tagName, blob):
    from bs4 import BeautifulSoup
    parsedQuestion = BeautifulSoup(blob, "html.parser")
    thinkTag = parsedQuestion.find(tagName)
    if thinkTag:
        thinkTag.decompose()
    return parsedQuestion.prettify()

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openApiClient.chat.completions.create(model="gpt-oss:20b", messages=messages)
    responseContent = response.choices[0].message.content
    return parseResponseByRemovingTag("think", responseContent)

In [ ]:
gr.ChatInterface(chat, type="messages").launch(share=True)

In [ ]:
gr.close_all()